# VAE-DiT TTS — Inference Demo

Multilingual zero-shot TTS with voice cloning.

**Features:**
- 🎙️ Clone any voice with 3-10s reference audio
- 🌏 Multilingual: ZH / JA / EN
- ⚡ 30-step flow matching inference
- 🎵 48kHz high-fidelity output
- ✂️ Auto sentence splitting for long text

## 1. Install Dependencies

In [ ]:
!pip install -q torch torchaudio transformers diffusers huggingface_hub
!pip install -q phonemizer pykakasi pypinyin rjieba bitsandbytes
!apt-get install -qq espeak-ng > /dev/null 2>&1 || brew install espeak-ng 2>/dev/null || echo 'Please install espeak-ng manually'

## 2. Download Model from HuggingFace

In [ ]:
from huggingface_hub import hf_hub_download, snapshot_download
import sys

HF_REPO_ID = "Regeny/VAE-DiT-TTS"

print("Downloading model files...")
CHECKPOINT_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="checkpoint.pt")
VOCAB_PATH = hf_hub_download(repo_id=HF_REPO_ID, filename="char_vocab.json")

CODE_DIR = snapshot_download(
    repo_id=HF_REPO_ID,
    allow_patterns=["*.py", "*.yaml", "*.json"],
    ignore_patterns=["*.pt", "*.bin", "*.safetensors"],
)

VAE_DIR = snapshot_download(
    repo_id=HF_REPO_ID,
    allow_patterns=["vae/*"],
)

if CODE_DIR not in sys.path:
    sys.path.insert(0, CODE_DIR)

print(f"Checkpoint: {CHECKPOINT_PATH}")
print("✅ Download complete!")

## 3. Load Models

In [ ]:
import torch
from IPython.display import Audio, display

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

from inference import load_checkpoint
dit, text_encoder, dur_pred, flow, cfg, char_tokenizer = load_checkpoint(
    CHECKPOINT_PATH, device, vocab_path_override=VOCAB_PATH
)
print(f"DiT: {dit.num_params / 1e6:.1f}M params")

from models.vae import load_vae, vae_encode, vae_decode
VAE_PATH = f"{VAE_DIR}/vae"
vae = load_vae(VAE_PATH, device=device, precision="fp16")
print("✅ All models loaded!")

## 4. Configuration

In [ ]:
# ============================================================
# 🎙️ Reference Audio
# ============================================================
PROMPT_AUDIO = "prompt.wav"          # 3-10s reference audio
PROMPT_TEXT  = "参考音频的文字内容"     # Transcription
PROMPT_LANG  = "ZH"                  # ZH / JA / EN

# ============================================================
# 📝 Text to Synthesize
# ============================================================
TTS_TEXT = "你好，今天天气真好。我们一起出去玩吧！明天也是好天气呢。"
TTS_LANG = "ZH"

# ============================================================
# ⚙️ Inference Parameters
# ============================================================
CFG_SCALE = None         # CFG scale (default: 3.0)
N_STEPS   = None         # Sampling steps (default: 30)
DURATION  = None         # Duration in seconds (None = auto)
SEED      = 42

# ✂️ Sentence splitting: automatically split long text by punctuation
# Recommended for long sentences to avoid missing words
SPLIT_SENTENCES = True

## 5. Generate Speech

In [ ]:
from inference import inference, inference_long

output_path = "output.wav"
infer_fn = inference_long if SPLIT_SENTENCES else inference

gen_latent = infer_fn(
    dit, text_encoder, dur_pred, flow, cfg,
    prompt_audio_path=PROMPT_AUDIO,
    prompt_text=PROMPT_TEXT,
    tts_text=TTS_TEXT,
    prompt_language=PROMPT_LANG,
    tts_language=TTS_LANG,
    char_tokenizer=char_tokenizer,
    vae_encode_fn=lambda wav: vae_encode(vae, wav),
    vae_decode_fn=lambda lat: vae_decode(vae, lat),
    output_path=output_path,
    duration=DURATION,
    cfg_scale=CFG_SCALE,
    n_steps=N_STEPS,
    seed=SEED,
)

print(f"\n✅ Generated: {output_path}")

## 6. Play Audio

In [ ]:
print("🔊 Synthesized Audio")
print(f"   Text: {TTS_TEXT}")
display(Audio(output_path))

print("\n🎙️ Reference Audio")
display(Audio(PROMPT_AUDIO))

---

## 7. Test Sentence Splitting

In [ ]:
from inference import split_text

test_cases = [
    "短句不切",
    "你好，今天天气真好。我们一起出去玩吧！",
    "今天是星期一，明天是星期二。后天是星期三！大后天呢？那就是星期四了。",
    "Hello, how are you? I'm fine, thank you. And you?",
]

for text in test_cases:
    segs = split_text(text)
    print(f"\nInput:  {text}")
    print(f"Output: {segs}")

## 8. Batch Generation (Optional)

In [ ]:
test_texts = [
    ("你好，今天天气真好", "ZH"),
    ("我们一起去吃饭吧", "ZH"),
    ("明天你有空吗？我想和你聊聊", "ZH"),
]

for i, (text, lang) in enumerate(test_texts):
    out = f"output_{i}.wav"
    print(f"\n[{i+1}/{len(test_texts)}] {text}")
    inference_long(
        dit, text_encoder, dur_pred, flow, cfg,
        prompt_audio_path=PROMPT_AUDIO,
        prompt_text=PROMPT_TEXT,
        tts_text=text,
        prompt_language=PROMPT_LANG,
        tts_language=lang,
        char_tokenizer=char_tokenizer,
        vae_encode_fn=lambda wav: vae_encode(vae, wav),
        vae_decode_fn=lambda lat: vae_decode(vae, lat),
        output_path=out,
        seed=SEED,
    )
    display(Audio(out))